# Pandas Profiling: a full EDA report in one line

Run each cell in order with **Shift + Enter**. Each step matches a section of the Note.
The data (`data/titanic_train.csv`) is Kaggle's Titanic `train.csv`: 891 passengers.
The report is saved next to this notebook as `titanic_report.html`; open it in any web browser.

## 1. Load the data

In [ ]:
import pandas as pd

df = pd.read_csv("data/titanic_train.csv")   # read the Titanic table
df.head()                                     # the first 5 rows

## 2. Install the library

The library has been renamed twice: `pandas-profiling` (until 2023), then `ydata-profiling`,
and since 2026 `fg-data-profiling`, imported as `data_profiling`.
Install it once from a terminal (or in a cell, with `!` in front):

```
pip install fg-data-profiling
```

### 2.1 Make it work with pandas 3

fg-data-profiling 4.20 officially supports pandas below 3. It runs on pandas 3 with one change:
pandas 3 stores text in Arrow arrays, which the library cannot add up, so we ask for plain Python strings.
With pandas 2 this cell is not needed.

In [ ]:
# Store text columns as plain Python strings (must run before the report is built)
pd.set_option("mode.string_storage", "python")

## 3. Build the report and save it as a web page

In [ ]:
from data_profiling import ProfileReport

# Build the report object from the DataFrame.
# By default the Correlations section shows one "Auto" heatmap;
# we also switch on Pearson's r, the coefficient from the Note on understanding data.
profile = ProfileReport(
    df,
    title="Titanic passengers",
    correlations={"auto": {"calculate": True}, "pearson": {"calculate": True}},
    progress_bar=False,
)

# Write it to an HTML file in this folder
profile.to_file("titanic_report.html")

## 4. The numbers behind the report

The report is built from a description object. Printing a few of its numbers
checks them against the Overview section (and against the earlier Notes).

In [ ]:
desc = profile.get_description()       # everything the report shows, as Python objects
table = desc.table                      # the Overview statistics

print("Variables (columns):     ", table["n_var"])
print("Observations (rows):     ", table["n"])
print("Missing cells:           ", table["n_cells_missing"])
print("Missing cells (%):       ", round(table["p_cells_missing"] * 100, 1))
print("Duplicate rows:          ", table["n_duplicates"])
print("Variable types:          ", table["types"])

In [ ]:
# The Alerts list: each one names a column and a possible problem
for alert in desc.alerts:
    print(alert)

## 5. A lighter report for big data

On a large dataset the full report is slow. `minimal=True` skips the costly parts
(correlations, interactions and the missing-value charts) and keeps the rest:

```python
ProfileReport(df, minimal=True).to_file("report_minimal.html")
```

## 6. Viewing the report inside the notebook

`profile.to_notebook_iframe()` shows the same report inside Jupyter.
We leave it commented out: it stores the whole 4-5 MB page in the notebook file.

In [ ]:
# profile.to_notebook_iframe()